# 용늪 21–0 ka 생물지형학 실습
### BIOME4–Pelletier 결합모델

용늪의 식생, 토양, 지형이 장기간에 걸쳐 상호작용하는 과정을 계산한다. 기후와 지형발달 계수를 변경하여 정적 지형과 동적 지형의 결과를 비교한다.

**계산 범위**  
21.0–0.0 ka BP, 0.1 kyr 간격(211시점), 20 m 격자(유효 298셀). 기본 기후는 CHELSA–TraCE21k이며 식생에는 BIOME4 v4.2b2, 지형발달에는 Pelletier 계열 수치모델을 사용한다.

**실습 순서**  
환경 설정 → 자료 복원 → 실험조건 변경 → 모델 실행 → 시계열 및 분포도 확인 → 실험 간 비교.

기본 설정은 **21–0 ka 전체기간**이다. 계산 시간을 줄여 기능을 확인하려면 21–20 ka를 선택할 수 있다.

기온과 강수의 조정값은 민감도 실험을 위한 가정이며, 실제 과거 기후대를 복원한 값이 아니다. 사면수송과 토양생산 매개변수의 변화 역시 별도의 검증 없이 실측 계수로 해석하지 않는다.

**자료 출처**  
[용늪 PB4Studio CHELSA21K 원본](https://github.com/physical-geogrpahy-forever/yongneup_nb/tree/main/pb4_chelsa21k) (모델 및 입력자료를 이 노트북에 포함). 실행 중 기후·토양·지형자료 다운로드는 필요하지 않다.

In [ ]:
#@title 1. Google Colab 환경 준비 (최초 한 번 실행)
import os, sys, subprocess, importlib.util
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OMP_DYNAMIC"] = "FALSE"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
if subprocess.run(["bash","-lc","command -v gfortran"],capture_output=True).returncode != 0:
    subprocess.check_call(["apt-get","update","-qq"])
    subprocess.check_call(["apt-get","install","-y","-qq","gfortran"])
required={"numpy":"numpy","pandas":"pandas","matplotlib":"matplotlib","rasterio":"rasterio","scipy":"scipy","xarray":"xarray","geopandas":"geopandas>=1.0","shapely":"shapely>=2.0","pyogrio":"pyogrio","pyproj":"pyproj","netCDF4":"netCDF4"}
missing=[spec for module,spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
# Linux/Colab 그래프의 한글 식생유형명이 네모로 표시되지 않도록 글꼴 확인
import matplotlib.font_manager as fm
korean_fonts={"NanumGothic","Noto Sans CJK KR","Noto Sans CJK JP","Malgun Gothic"}
if not (korean_fonts & {font.name for font in fm.fontManager.ttflist}):
    subprocess.check_call(["apt-get","install","-y","-qq","fonts-nanum"])
    font_file="/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
    if os.path.isfile(font_file):
        fm.fontManager.addfont(font_file)
# Colab 기본 글꼴에는 한국어가 없을 수 있으므로 한국어 글꼴을 등록
from pathlib import Path
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
ko_fonts=("NanumGothic","Noto Sans CJK KR","Malgun Gothic","AppleGothic")
installed={ft.name for ft in fm.fontManager.ttflist}
if not any(name in installed for name in ko_fonts):
    subprocess.check_call(["apt-get","update","-qq"])
    subprocess.check_call(["apt-get","install","-y","-qq","fonts-nanum"])
    for path in Path("/usr/share/fonts").rglob("NanumGothic*.ttf"):
        fm.fontManager.addfont(str(path))
    installed={ft.name for ft in fm.fontManager.ttflist}
font_name=next((name for name in ko_fonts if name in installed),None)
if font_name:
    plt.rcParams["font.family"]=font_name
plt.rcParams["axes.unicode_minus"]=False
print("Fortran, Python, 글꼴 준비 완료:",font_name or "기본 글꼴")

## 2. 입력자료

용늪 DEM, 토심, 토성, 월별 기후 및 BIOME4 소스는 다음 셀에 포함되어 있다. 해당 셀은 자료 복원용이므로 수정하지 않는다. 코드를 보려면 Colab에서 **코드 표시**를 선택한다.

In [ ]:
#@title 2. 내장 자료 복원
import base64, io, zipfile, hashlib, sys
from pathlib import Path
ARCHIVE_BASE64 = """
...
BgAAAAA4ADgA9BYAAAtcAwAAAA==
"""
raw=base64.b64decode("".join(ARCHIVE_BASE64.split()))
checksum=hashlib.sha256(raw).hexdigest()
EXPECTED="93790ba804a9cbce01291015af2750974d85b9688f0894de8d51d46cfb5a4b7b"
print("내장 모델 SHA256:", checksum)
if checksum != EXPECTED:
    raise RuntimeError("내장 모델 해시와 GitHub 체크섬이 일치하지 않습니다: 원본을 확인하세요.")
ROOT=Path("/content/yongneup_biome4_teaching") if Path("/content").exists() else Path.cwd()/"yongneup_biome4_teaching"
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    archive.extractall(ROOT)
MODEL_DIR=ROOT/"PB4Studio_v6.6.3_CHELSA21K"
assert (MODEL_DIR/"pb4studio"/"runner.py").is_file()
assert (MODEL_DIR/"fortran_src"/"biome4_original_4_2b2.f").is_file()
assert (MODEL_DIR/"embedded_inputs"/"yongneup_exact20m"/"YONGNEUP_CHELSA_TRACE21k_ENVICLOUD_RAW_WIDE.csv").is_file()
if str(MODEL_DIR) not in sys.path: sys.path.insert(0,str(MODEL_DIR))
from pb4studio.config import RunConfig, RunMode, InputPaths, TimeConfig, ScienceConfig
from pb4studio import runner
print("모델 및 입력자료 복원 완료")

## 3. 실험조건 설정

기본값은 21–0 ka 전체기간의 용늪 실험이다. **실험 이름**을 바꾸면 서로 다른 설정의 결과를 구분하여 저장할 수 있다.

기온 편차는 월별 최저·최고기온에 동일하게 더하며, 강수량 배율은 각 월의 강수에 곱한다. 온난/한랭은 ±4°C, 습윤/건조는 강수량 ×1.5/×0.5이다. 이는 원래 CHELSA 시계열에 일정한 편차를 적용한 **강화 민감도 시나리오**로서 실제 고기후 복원치가 아니다.

In [ ]:
#@title 3. 실험조건
experiment_name = "baseline" #@param {type:"string"}
period = "21–0 ka (전체)" #@param ["21–0 ka (전체)", "21–20 ka (짧은 실습)", "21–18 ka", "21–15 ka"]
climate_type = "CHELSA 원자료" #@param ["CHELSA 원자료", "한랭건조", "한랭습윤", "온난건조", "온난습윤"]
temperature_offset_c = 0.0 #@param {type:"slider", min:-5, max:5, step:0.5}
precipitation_multiplier = 1.0 #@param {type:"slider", min:0.5, max:1.5, step:0.05}
initial_soil_depth_multiplier = 1.0 #@param {type:"slider", min:0.5, max:1.5, step:0.1}
uplift_m_per_kyr = 0.08 #@param {type:"slider", min:0.0, max:0.16, step:0.02}
soil_production_a = 0.037 #@param {type:"slider", min:0.02, max:0.06, step:0.001}
hillslope_eemt_c = 0.033 #@param {type:"slider", min:0.015, max:0.06, step:0.001}
hillslope_biomass_d = 0.050 #@param {type:"slider", min:0.02, max:0.10, step:0.005}
fluvial_erodibility_k0 = 0.020 #@param {type:"slider", min:0.01, max:0.04, step:0.002}

# 수치적분의 임계경사 Sc=1.50과 0.1 kyr 결합간격은 기본적으로 고정합니다.
# 이는 학생이 건드리는 물리적 영향 계수와 수치안정성 설정을 구분하기 위한 조치입니다.
print("실험 이름:", experiment_name, "| 기간:",period, "| 가상 기후:",climate_type)

In [ ]:
#@title 4. 가상 기후와 초기 토심을 원본 자료의 복사본에 적용
import re, shutil, json, numpy as np, pandas as pd, rasterio
from dataclasses import asdict
assert re.fullmatch(r"[A-Za-z0-9_-]{1,40}", experiment_name), "실험 이름은 영문자, 숫자, _ 또는 -로 1~40자입니다."
period_map={"21–20 ka (짧은 실습)":20.0, "21–18 ka":18.0,"21–15 ka":15.0,"21–0 ka (전체)":0.0}
end_ka=period_map[period]
climate_presets={"CHELSA 원자료":(0.0,1.0),"한랭건조":(-4.0,0.5),"한랭습윤":(-4.0,1.5),"온난건조":(4.0,0.5),"온난습윤":(4.0,1.5)}
base_dT, base_P=climate_presets[climate_type]
dT=base_dT+float(temperature_offset_c)
p_factor=base_P*float(precipitation_multiplier)
assert -10 < dT < 10 and 0.2 < p_factor < 2.5
orig_inputs=MODEL_DIR/"embedded_inputs"/"yongneup_exact20m"
work=ROOT/"teaching_experiments"/experiment_name
inputs=work/"inputs"
if inputs.exists(): shutil.rmtree(inputs)
shutil.copytree(orig_inputs,inputs)
rawcsv=inputs/"YONGNEUP_CHELSA_TRACE21k_ENVICLOUD_RAW_WIDE.csv"
clim=pd.read_csv(rawcsv)
for month in range(1,13):
    mm=f"{month:02d}"
    for key in (f"tasmin_raw_{mm}",f"tasmax_raw_{mm}"):
        assert key in clim.columns,key
        clim[key]=clim[key].astype(float)+dT # Kelvin offset and Celsius offset have the same increment
    key=f"pr_raw_{mm}"
    assert key in clim.columns,key
    clim[key]=clim[key].astype(float)*p_factor
clim.to_csv(rawcsv,index=False,float_format="%.6f")
assert len(clim)==211 and clim.notna().all().all(), "21 ka 기후자료 범위를 확인하세요."
soil_path=inputs/"soil_depth_20m_exact_m.tif"
with rasterio.open(soil_path) as src:
    soil_masked=src.read(1, masked=True)
    profile=src.profile.copy()
    soil_array=np.asarray(soil_masked.filled(src.nodata if src.nodata is not None else -9999),dtype="float32")
    mask=np.ma.getmaskarray(soil_masked)
    soil_array[~mask]*=float(initial_soil_depth_multiplier)
with rasterio.open(soil_path,"w",**profile) as dst: dst.write(soil_array,1)
print(f"월평균기온 변화 {dT:+.1f}°C / 월강수 {p_factor:.2f}배 / 초기 토심 {initial_soil_depth_multiplier:.2f}배")
print("원본은 변형하지 않고 실험용 입력 복사본만 수정했습니다.")

## 5. 모델 실행

동일한 기후에서 **정적 지형**(초기 지형·토심 고정)과 **동적 지형**(토양생산·사면수송·유수침식·융기 계산)을 비교한다. 식생은 각 0.1 kyr 시점에서 BIOME4로 계산한다. 지형발달 계산에는 안정성을 위한 적응형 세부 시간간격을 사용한다.

기본 실행은 21–0 ka의 211개 식생 시점과 210개 지형발달 구간을 포함한다. 계산 시간을 줄이고 싶으면 3번 셀에서 기간을 21–20 ka로 변경한다.

In [ ]:
#@title 5. 정적·동적모델 실행
import time
from pb4studio.pelletier_geomorph import PelletierStrictConfig
out_dir=work/"output"
# 같은 실험 이름을 재실행하더라도 과거 실행 결과가 혼입되지 않게 정리
if out_dir.exists(): shutil.rmtree(out_dir)
out_dir.mkdir(parents=True,exist_ok=True)
# 정확한 0.1 kyr 모형 시점에서만 스냅샷을 저장합니다.
times=np.round(np.arange(21.0,end_ka-0.001,-0.1),6)
assert len(times)>1 and abs(float(times[-1])-end_ka)<1e-5
snaps=(21.0,15.0,10.0,5.0,0.0) if end_ka == 0.0 else (float(times[0]),float(times[len(times)//2]),float(times[-1]))
cfg=RunConfig(
    mode=RunMode.COMPARISON,
    inputs=InputPaths(
        dem_path=str(inputs/"dem_20m_exact.tif"),
        climate_nc_path=str(rawcsv),
        soil_depth_path=str(soil_path),
        soil_texture_path=str(inputs/"soil_texture_20m_exact.tif"),
        basin_polygon_path=str(inputs/"basin_exact20m.geojson"),
        open_free_shp=str(inputs/"outlet_open_free_exact20m.geojson"),
        open_fixed_slope_shp=None,
        soil_depth_unit="m",
        validation_points_csv="", # 짧은 수업 실험에서는 Jang 평가가 아니라 물리반응에 초점
    ),
    time=TimeConfig(start_ka=21.0,end_ka=float(end_ka),interval_kyr=0.1,snapshot_ka=snaps),
    resolution_m_override=20.0,
    requested_core_count=2,
    pelletier=PelletierStrictConfig(
        uplift_m_per_kyr=float(uplift_m_per_kyr),
        p0_base_m_per_kyr=float(soil_production_a),
        kd_c_eemt_m_per_kyr_per_mj=float(hillslope_eemt_c),
        kd_d_agb_m_per_kyr_per_kgm2=float(hillslope_biomass_d),
        k0_regolith=float(fluvial_erodibility_k0),
        hillslope_critical_slope=1.50,
    ),
    science=ScienceConfig(
        biome4_variant="mckenzie2003",
        force_compile_biome4=False,
        validation_accuracy_method="per_output_time",
        basin_presence_min_fraction=0.01,
    ),
    output_dir=str(out_dir),
)
cfg.to_json(work/"run_config.json")
(work/"teaching_settings.json").write_text(json.dumps({
    "climate_type":climate_type,"dT_C":dT,"precip_multiplier":p_factor,
    "soil_depth_multiplier":initial_soil_depth_multiplier,"end_ka":end_ka,
    "u_m_per_kyr":uplift_m_per_kyr, "p0_a":soil_production_a,
    "kd_c":hillslope_eemt_c,"kd_d":hillslope_biomass_d,"K0":fluvial_erodibility_k0,
    "model_sha256":checksum,
},ensure_ascii=False,indent=2),encoding="utf-8")
def progress(case_label,step_index,n_steps,ka):
    if step_index in (0,n_steps-1) or step_index%10==0:
        print(f"[{case_label}] {step_index+1}/{n_steps}, {ka:.1f} ka BP",flush=True)
import warnings
start=time.perf_counter()
# 유역 밖 NoData 셀에 대한 nanmean 경고는 계산에 영향을 주지 않음
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning, module=r"pb4studio\.runner")
    results=runner.run(cfg,progress_callback=progress)
print(f"실행 완료 | 경과 시간: {time.perf_counter()-start:.1f}s | 저장 위치: {out_dir}")
print("BIOME4–Pelletier 모의 완료")

In [ ]:
#@title 6. 정적·동적 시계열 비교
import matplotlib.pyplot as plt
summaries={}
for label,subdir in [("Static","model_static"),("Dynamic","model_dynamic")]:
    f=out_dir/subdir/"summary_timeseries.csv"
    assert f.is_file(),f
    summaries[label]=pd.read_csv(f)

fig,ax=plt.subplots(1,3,figsize=(15,4.5),constrained_layout=True)
for axis,(col,ylabel) in zip(ax,[
    ("mean_elevation_m","Mean elevation (m)"),
    ("mean_soil_depth_m","Mean soil depth (m)"),
    ("mean_npp","Mean NPP (gC m$^{-2}$ yr$^{-1}$)")]):
    for label,color in [("Static","royalblue"),("Dynamic","firebrick")]:
        df=summaries[label]
        assert col in df.columns,(col,list(df.columns))
        axis.plot(df["ka_bp"],df[col],lw=2,label=label,color=color)
    axis.invert_xaxis()
    axis.set_xlabel("Age (ka BP)")
    axis.set_ylabel(ylabel)
    axis.legend(frameon=False)
plt.show()
print("최종시점 통계")
for label,df in summaries.items():
    last=df.iloc[-1]
    print(label, "ka",last["ka_bp"],"고도",round(last["mean_elevation_m"],3),"m, 토심",round(last["mean_soil_depth_m"],3),"m, NPP",round(last["mean_npp"],2))

## 7. 시기별 공간분포와 식생 변화
21–0 ka 전체기간을 실행하면 **21, 15, 10, 5, 0 ka**의 토심·식생·고도·NPP 지도를 확인할 수 있다. 각 행은 같은 시점의 **동적모델 / 정적모델 / 두 모델의 차이**다.

식생유형 지도 오른쪽의 녹색은 두 모델의 판정이 일치한 격자이고, 붉은색은 불일치한 격자다. 식생유형 코드 간 뺄셈은 의미가 없으므로 수치 차이 지도로 해석하지 않는다. 토심·고도·NPP 지도는 모델 간 차이를 실제 단위로 표시한다.

다음 절의 식생 면적비율 그래프는 지도에 표시된 다섯 시점만이 아니라 **0.1 kyr 간격의 전체 시계열**을 사용한다.


In [ ]:
#@title 7. 시기별 토심·식생·고도·NPP 지도 및 식생 비율 변화
from IPython.display import display, Image, Markdown
figure_dir = out_dir / "figures"
panels = [
    ("시기에 따른 토심 분포 — 동적 / 정적 / 차이", "snapshot_dynamic_static_soil_depth_m.png"),
    ("시기에 따른 식생유형 — 동적 / 정적 / 일치·불일치", "snapshot_dynamic_static_vegetation_code.png"),
    ("시기에 따른 고도 분포 — 동적 / 정적 / 차이", "snapshot_dynamic_static_elevation_m.png"),
    ("시기에 따른 순일차생산량(NPP) — 동적 / 정적 / 차이", "snapshot_dynamic_static_npp.png"),
]
for caption, filename in panels:
    p = figure_dir / filename
    if not p.is_file():
        raise FileNotFoundError(f"{p} — 5번 모델 실행 셀부터 실행하십시오.")
    display(Markdown("### " + caption))
    display(Image(filename=str(p)))
print("그림 저장 경로:", figure_dir)


## 8. 전체기간 수치와 식생 면적비율 비교
이하의 비교표는 각 시점에서 **동적모델 − 정적모델**을 계산한다. 고도·토심·NPP의 평균뿐 아니라 유역 내 표준편차·최소·최대, LAI, 증발산, 유출량, EEMT 및 모델에서 산출한 지상부 바이오매스 추정지표(AGB*)를 함께 저장한다.

식생은 범주형 자료이므로 우점유형의 평균을 계산하지 않고, 유효 격자 수를 분모로 한 **식생유형별 면적비율(%)**로 비교한다. 20 m 격자의 셀 면적은 동일하다.

아래 화면에는 선정 시점만 표시하지만, CSV에는 21–0 ka의 **211시점 전체**가 저장된다.


In [ ]:
#@title 8. 전 시점 정적·동적 비교표와 식생 면적비율 CSV
import numpy as np
import pandas as pd
from IPython.display import display
report_dir = out_dir / "student_report"
report_dir.mkdir(parents=True, exist_ok=True)
selected_ages = tuple(float(x) for x in snaps)
VEG_NAMES = {0:"침엽수림",1:"활엽수림",2:"혼효림",3:"초본·개방식생",4:"무식생·나지"}
CASE_DIRS = {"정적":"model_static","동적":"model_dynamic"}
source_data = {
    label:pd.read_csv(out_dir/sub/"summary_timeseries.csv")
    for label,sub in CASE_DIRS.items()
}
metric_names = [
    "mean_elevation_m","sd_elevation_m","min_elevation_m","max_elevation_m",
    "mean_soil_depth_m","sd_soil_depth_m","min_soil_depth_m","max_soil_depth_m",
    "mean_npp","sd_npp","min_npp","max_npp",
    "mean_lai","sd_lai","mean_aet_mm_yr","mean_runoff_mm_yr",
    "mean_eemt","reich_lai_sapwood_agb_mean_kg_m2"
]
shared = set(source_data["정적"].columns) & set(source_data["동적"].columns)
metric_names = [c for c in metric_names if c in shared]
assert {"mean_elevation_m","mean_soil_depth_m","mean_npp"}.issubset(metric_names)
for label,df in source_data.items():
    assert df["ka_bp"].is_unique, f"{label}: duplicated time step"
sta = source_data["정적"][["ka_bp"]+metric_names].rename(
    columns={c:c+"_static" for c in metric_names})
dyn = source_data["동적"][["ka_bp"]+metric_names].rename(
    columns={c:c+"_dynamic" for c in metric_names})
full_comparison = sta.merge(dyn,on="ka_bp",validate="one_to_one").sort_values("ka_bp",ascending=False)
for c in metric_names:
    full_comparison[c+"_dynamic_minus_static"] = full_comparison[c+"_dynamic"]-full_comparison[c+"_static"]
full_comparison.to_csv(report_dir/"01_full_timeseries_static_dynamic.csv",index=False,encoding="utf-8-sig")
age_mask = np.isclose(full_comparison["ka_bp"].to_numpy()[:,None],
                      np.asarray(selected_ages)[None,:],atol=1e-6).any(axis=1)
selected_comparison = full_comparison.loc[age_mask]
selected_comparison.to_csv(report_dir/"02_selected_ages_comparison.csv",index=False,encoding="utf-8-sig")
display(selected_comparison[[
    "ka_bp",
    "mean_soil_depth_m_static","mean_soil_depth_m_dynamic","mean_soil_depth_m_dynamic_minus_static",
    "sd_soil_depth_m_static","sd_soil_depth_m_dynamic",
    "mean_npp_static","mean_npp_dynamic","mean_npp_dynamic_minus_static",
    "mean_elevation_m_static","mean_elevation_m_dynamic","mean_elevation_m_dynamic_minus_static"
]].round(3))

def parse_class_counts(value):
    counts = {code:0 for code in VEG_NAMES}
    for token in str(value).split(";"):
        if ":" in token:
            code,count = token.strip().split(":",1)
            if int(code) in counts:
                counts[int(code)] += int(float(count))
    return counts

ratio_records = []
for label,df in source_data.items():
    for row in df.itertuples(index=False):
        n = int(row.n_cells)
        counts = parse_class_counts(row.vegetation_class_counts)
        for code,name in VEG_NAMES.items():
            ratio_records.append({
                "model":label,"ka_bp":float(row.ka_bp),"vegetation_code":code,
                "vegetation_type":name,"cells":counts[code],
                "area_m2":counts[code]*float(row.area_m2)/n,
                "area_percent":100.0*counts[code]/n
            })
vegetation_ratios = pd.DataFrame(ratio_records)
vegetation_ratios.to_csv(report_dir/"03_vegetation_ratio_all_times.csv",index=False,encoding="utf-8-sig")
ratio_age_mask=np.isclose(vegetation_ratios["ka_bp"].to_numpy()[:,None],
                         np.asarray(selected_ages)[None,:],atol=1e-6).any(axis=1)
ratio_selected=vegetation_ratios.loc[ratio_age_mask].copy()
ratio_selected.to_csv(report_dir/"04_selected_ages_vegetation_ratio.csv",index=False,encoding="utf-8-sig")
display(ratio_selected.pivot_table(index=["ka_bp","model"],columns="vegetation_type",
    values="area_percent",aggfunc="first").sort_index(ascending=False).round(2))
# 원래 자동 그림에는 '1000년 간격'이라고 잘못 쓰인 축 제목이 있어 0.1 kyr로 교정한다.
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pb4studio.validation import setup_korean_matplotlib
setup_korean_matplotlib()
veg_palette = {0:"#2c7fb8",1:"#2ca02c",2:"#addd8e",3:"#fdbf6f",4:"#bdbdbd"}
fig,axs=plt.subplots(2,1,figsize=(11,7),sharex=True)
for ax,model in zip(axs,("동적","정적")):
    sub=vegetation_ratios[vegetation_ratios["model"]==model]
    wide=sub.pivot_table(index="ka_bp",columns="vegetation_code",
                         values="area_percent",aggfunc="first").sort_index()
    x=wide.index.to_numpy()
    y=[wide.get(k,pd.Series(0,index=wide.index)).to_numpy() for k in VEG_NAMES]
    ax.stackplot(x,*y,colors=[veg_palette[k] for k in VEG_NAMES])
    ax.set_ylabel("Area ratio (%)")
    ax.set_title(model+" model")
    ax.set_ylim(0,100)
    ax.grid(alpha=.2)
axs[-1].invert_xaxis()
axs[-1].set_xlabel("Age (ka BP; 0.1 kyr interval)")
fig.legend(handles=[Patch(facecolor=veg_palette[k],label=VEG_NAMES[k]) for k in VEG_NAMES],
           loc="lower center",bbox_to_anchor=(.5,.01),ncol=5,frameon=False)
fig.tight_layout(rect=(0,.07,1,1))
ratio_png=out_dir/"figures"/"timeseries_vegetation_ratio.png"
fig.savefig(ratio_png,dpi=170,bbox_inches="tight")
plt.show()
print("식생유형 면적비율 그래프:",ratio_png)
print("전체 시계열:",len(full_comparison),"시점 | 식생 비율:",len(vegetation_ratios),"행")
print("표와 CSV:",report_dir)


## 8A. BIOME4의 13개 PFT별 생육 반응

**PFT 01–13**의 잠재 NPP와 LAI, 기후 생육 제약 충족 격자 비율을 21–0 ka 전 기간에 걸쳐 비교한다. 원본 Fortran BIOME4의 PFT 01은 비활성화되어 있다. 각 PFT의 잠재 NPP를 모두 합하거나 생육 제약 충족 비율을 합하여 실제 식생 생산량 또는 면적비율로 해석하지 않는다. **실제 식생유형별 면적비율은 앞 절의 5개 축약 식생유형 결과**를 사용한다.

출력 그래프는 정적·동적 각각 세 종류이며, NPP·LAI·생육 제약 충족 비율을 모든 PFT에 대해 개별 축으로 보여준다.


In [ ]:
#@title 8A. 13개 PFT별 NPP·LAI·생육 제약 충족률
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
PFT_NAMES={
    1:"열대 상록활엽수",2:"열대 낙엽활엽수",3:"온대 상록활엽수",
    4:"온대 낙엽활엽수",5:"온대 상록침엽수",6:"아한대 상록수",
    7:"아한대 낙엽수",8:"온대 초본",9:"열대·난온대 초본",
    10:"건조지 목본",11:"툰드라 관목",12:"한랭 초본",13:"지의류·광엽초본"}
PFT_METRICS={
    "NPP":("npp_mean","잠재 NPP (gC m⁻² yr⁻¹)"),
    "LAI":("lai_mean","잠재 LAI"),
    "PASS":("constraint_pass_fraction","생육 제약 충족 격자 비율")}
pft_rows=[]
for model_name,df in source_data.items():
    for row in df.itertuples(index=False):
        for pft_no,pft_name in PFT_NAMES.items():
            pft_prefix=f"pft{pft_no:02d}_"
            pft_rows.append({
                "model":model_name,"ka_bp":float(row.ka_bp),
                "pft_id":pft_no,"pft_name":pft_name,
                "potential_npp":getattr(row,pft_prefix+"npp_mean"),
                "potential_lai":getattr(row,pft_prefix+"lai_mean"),
                "constraint_pass_fraction":getattr(row,pft_prefix+"constraint_pass_fraction")})
pft_long=pd.DataFrame(pft_rows)
pft_long.to_csv(report_dir/"13_pft_all_times.csv",index=False,encoding="utf-8-sig")
pft_long[np.isclose(pft_long.ka_bp,float(end_ka))].to_csv(
    report_dir/"14_pft_end_age.csv",index=False,encoding="utf-8-sig")
for model_name in source_data:
    subset=pft_long[pft_long.model==model_name]
    for metric,(column,ylabel) in PFT_METRICS.items():
        fig,axes=plt.subplots(4,4,figsize=(14.5,11),sharex=True,constrained_layout=True)
        for pid,label in PFT_NAMES.items():
            a=axes.flat[pid-1]
            seq=subset[subset.pft_id==pid]
            y=seq[{"NPP":"potential_npp","LAI":"potential_lai","PASS":"constraint_pass_fraction"}[metric]]
            a.plot(seq.ka_bp,y,lw=1.5)
            a.set_title(f"PFT{pid:02d} {label}"+(" (비활성)" if pid==1 else ""),fontsize=9)
            a.grid(alpha=.2)
            if metric=="PASS":a.set_ylim(-.03,1.03)
            if (pid-1)%4==0:a.set_ylabel(ylabel,fontsize=8)
        for a in axes.flat[13:]:a.axis("off")
        axes[0,0].invert_xaxis()
        fig.suptitle(f"BIOME4 13개 PFT · {metric} · {model_name}",fontsize=14)
        out_png=figure_dir/f"pft_{metric.lower()}_{model_name.lower()}.png"
        fig.savefig(out_png,dpi=170,bbox_inches="tight")
        plt.show()
        print("생성:",out_png)
print("전체 PFT 진단 기록:",len(pft_long),"행 | CSV:",report_dir)


## 9. 격자별 차이와 식생 판정의 변화
두 모델의 평균 토심이 비슷하더라도 토양이 두꺼워진 격자와 얇아진 격자가 공존할 수 있다. 다음 표는 같은 위치의 20 m 격자를 직접 대조한 결과다.

**차이의 부호는 동적모델 − 정적모델**이다. 10백분위수(P10), 중앙값(P50), 90백분위수(P90), 평균 절대차이, 증가·감소한 격자 비율을 제공한다.

식생 일치율은 두 모델이 같은 식생유형을 판정한 격자의 비율이다. 식생 교차표는 **정적모델의 식생유형 → 동적모델의 식생유형**이 어떻게 달라졌는지 보여준다. 모델 간 차이이며, 시간에 따른 동일 격자의 실제 식생 전환과는 구분해야 한다.


In [ ]:
#@title 9. 시기별 격자 통계, 식생 일치율 및 식생 교차표
from pb4studio.snapshots import snapshot_dir_name
import rasterio
def raster_values(case,age,var):
    p = out_dir / CASE_DIRS[case] / snapshot_dir_name(float(age)) / (var+".tif")
    with rasterio.open(p) as ds:
        return ds.read(1,masked=True).filled(np.nan).astype(float)

agreement_rows,transition_rows,spatial_rows = [],[],[]
for age in selected_ages:
    sv=raster_values("정적",age,"vegetation_code")
    dv=raster_values("동적",age,"vegetation_code")
    mask=np.isfinite(sv)&np.isfinite(dv)
    n=int(mask.sum())
    assert n>0, f"{age} ka: no valid model cells"
    n_same=int(np.count_nonzero((sv==dv)&mask))
    agreement_rows.append({"ka_bp":age,"valid_cells":n,"same_cells":n_same,
        "different_cells":n-n_same,"agreement_percent":100*n_same/n})
    ct=pd.crosstab(sv[mask].astype(int),dv[mask].astype(int))
    for sc,sname in VEG_NAMES.items():
        for dc,dname in VEG_NAMES.items():
            count=int(ct.loc[sc,dc]) if sc in ct.index and dc in ct.columns else 0
            transition_rows.append({"ka_bp":age,"static_code":sc,"static_type":sname,
                "dynamic_code":dc,"dynamic_type":dname,
                "cells":count,"area_percent":100*count/n})
    for var,unit in [
        ("soil_depth_m","m"),("elevation_m","m"),
        ("npp","gC m-2 yr-1"),("lai","m2 m-2"),
        ("agb_reich_lai_sapwood","kg m-2 (AGB*)")
    ]:
        s=raster_values("정적",age,var)
        d=raster_values("동적",age,var)
        valid=np.isfinite(s)&np.isfinite(d)
        diff=d[valid]-s[valid]
        assert len(diff)>0
        spatial_rows.append({
            "ka_bp":age,"variable":var,"unit":unit,"valid_cells":len(diff),
            "static_mean":np.mean(s[valid]),"dynamic_mean":np.mean(d[valid]),
            "difference_mean":np.mean(diff),"difference_median":np.median(diff),
            "difference_p10":np.percentile(diff,10),"difference_p90":np.percentile(diff,90),
            "mean_absolute_difference":np.mean(np.abs(diff)),
            "dynamic_greater_percent":100*np.count_nonzero(diff>1e-8)/len(diff),
            "dynamic_less_percent":100*np.count_nonzero(diff< -1e-8)/len(diff)
        })
agreement_table=pd.DataFrame(agreement_rows)
transition_table=pd.DataFrame(transition_rows)
spatial_table=pd.DataFrame(spatial_rows)
agreement_table.to_csv(report_dir/"05_vegetation_agreement.csv",index=False,encoding="utf-8-sig")
transition_table.to_csv(report_dir/"06_vegetation_cross_tabulation.csv",index=False,encoding="utf-8-sig")
spatial_table.to_csv(report_dir/"07_spatial_difference_statistics.csv",index=False,encoding="utf-8-sig")
display(Markdown("### 정적·동적 식생 일치율"))
display(agreement_table.round(2))
display(Markdown("### 격자별 토심·고도·NPP 등의 차이"))
display(spatial_table[["ka_bp","variable","static_mean","dynamic_mean",
    "difference_mean","difference_p10","difference_median","difference_p90",
    "mean_absolute_difference","dynamic_greater_percent","dynamic_less_percent"]].round(3))
print("식생유형 교차표 및 전체 격자 통계 CSV 저장:",report_dir)


## 10. 기후·계수 변경 실험끼리 비교
먼저 `baseline`을 실행한 뒤, 설정 셀에서 기온·강수·토양생산·사면수송계수 중 **한 가지씩** 조정하고 실험 이름을 달리한다. 아래 비교에서는 **현재 실험과 종료시점이 같은 실험만** 사용한다.

정적·동적모델을 같은 환경에서 비교하는 것과, 서로 다른 기후·계수 실험을 비교하는 것은 별개의 비교다. 여러 매개변수를 동시에 바꾸면 단일 원인의 영향을 분리할 수 없다.


In [ ]:
#@title 10. 다른 실험과 같은 시점·같은 모델 기준 비교
from IPython.display import display
scenario_rows=[]
exp_root=ROOT/"teaching_experiments"
for exp in sorted(exp_root.iterdir()):
    conf=exp/"teaching_settings.json"
    if not conf.is_file(): continue
    options=json.loads(conf.read_text(encoding="utf-8"))
    if not np.isclose(float(options["end_ka"]),float(end_ka),atol=1e-6):
        continue
    for case_label,subdir in CASE_DIRS.items():
        file=exp/"output"/subdir/"summary_timeseries.csv"
        if not file.is_file():continue
        tab=pd.read_csv(file)
        for _,r in tab.iterrows():
            scenario_rows.append({
                "experiment":exp.name,"model":case_label,"ka_bp":float(r["ka_bp"]),
                "temperature_offset_C":options["dT_C"],
                "precipitation_multiplier":options["precip_multiplier"],
                "soil_multiplier":options["soil_depth_multiplier"],
                "uplift_m_per_kyr":options["u_m_per_kyr"],
                "soil_production_a":options["p0_a"],
                "hillslope_c":options["kd_c"],"hillslope_d":options["kd_d"],
                "fluvial_K0":options["K0"],
                "mean_elevation_m":r["mean_elevation_m"],
                "sd_elevation_m":r["sd_elevation_m"],
                "mean_soil_depth_m":r["mean_soil_depth_m"],
                "sd_soil_depth_m":r["sd_soil_depth_m"],
                "mean_npp":r["mean_npp"],"sd_npp":r["sd_npp"],
                "vegetation_class_counts":r["vegetation_class_counts"]
            })
scenario_timeseries=pd.DataFrame(scenario_rows)
assert not scenario_timeseries.empty, "먼저 모델 실행을 완료하세요."
scenario_timeseries.to_csv(report_dir/"08_experiment_timeseries.csv",index=False,encoding="utf-8-sig")
scenario_end=scenario_timeseries[np.isclose(scenario_timeseries["ka_bp"],float(end_ka),atol=1e-6)].copy()
if "baseline" in scenario_timeseries["experiment"].unique():
    base_rows=scenario_timeseries[scenario_timeseries["experiment"]=="baseline"]
    base_rows=base_rows[["model","ka_bp","mean_soil_depth_m","mean_npp","mean_elevation_m"]].rename(
        columns={k:k+"_baseline" for k in ("mean_soil_depth_m","mean_npp","mean_elevation_m")})
    scenario_end=scenario_end.merge(base_rows,on=["model","ka_bp"],how="left",validate="many_to_one")
    for v in ("mean_soil_depth_m","mean_npp","mean_elevation_m"):
        scenario_end[v+"_minus_baseline"]=scenario_end[v]-scenario_end[v+"_baseline"]
scenario_end.to_csv(report_dir/"09_experiment_end_comparison.csv",index=False,encoding="utf-8-sig")
# 대조실험과 동일한 시점·동일한 모델의 전체 시계열 차이를 별도로 저장
if "baseline" in scenario_timeseries["experiment"].unique():
    ref= scenario_timeseries[scenario_timeseries["experiment"]=="baseline"][
        ["model","ka_bp","mean_soil_depth_m","mean_npp","mean_elevation_m","sd_soil_depth_m","sd_npp","sd_elevation_m"]
    ].rename(columns={v:v+"_baseline" for v in (
        "mean_soil_depth_m","mean_npp","mean_elevation_m",
        "sd_soil_depth_m","sd_npp","sd_elevation_m")})
    scenario_all_comparison=scenario_timeseries.merge(ref,on=["model","ka_bp"],
        how="left",validate="many_to_one")
    for v in ("mean_soil_depth_m","mean_npp","mean_elevation_m",
              "sd_soil_depth_m","sd_npp","sd_elevation_m"):
        scenario_all_comparison[v+"_minus_baseline"] = (
            scenario_all_comparison[v]-scenario_all_comparison[v+"_baseline"])
    scenario_all_comparison.to_csv(report_dir/"10_full_experiment_vs_baseline.csv",
        index=False,encoding="utf-8-sig")

display(scenario_end.round(3))

import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(15,4.5),constrained_layout=True)
for ax,(col,ylabel) in zip(axes,[
    ("mean_soil_depth_m","Soil depth (m)"),
    ("mean_elevation_m","Elevation (m)"),
    ("mean_npp","NPP (gC m$^{-2}$ yr$^{-1}$)")
]):
    for (exp,case),group in scenario_timeseries.groupby(["experiment","model"]):
        ls="-" if case=="동적" else "--"
        ax.plot(group["ka_bp"],group[col],linestyle=ls,label=exp+" / "+case,lw=1.5)
    ax.invert_xaxis()
    ax.set_xlabel("Age (ka BP)")
    ax.set_ylabel(ylabel)
axes[0].legend(frameon=False,fontsize=8)
plt.show()
print("다중 실험 비교 CSV:",report_dir)


## 11. 서로 다른 실험의 공간분포 비교

기준실험을 실행한 뒤 다른 실험을 실행하면, 같은 시점·같은 모델에서 **기준실험 / 변경실험 / 차이**를 한눈에 비교할 수 있다. 토심은 미터 단위로 차이를 계산하고, 식생은 범주형 자료이므로 **일치·불일치 지도**로 비교한다.

기준실험의 이름과 확인할 시점을 지정한다. 지정한 시점의 지도가 두 실험 모두에 없을 경우, 공통으로 저장된 시점 중 마지막 시점을 사용한다. 실험기간이 서로 다른 결과는 비교하지 않는다.


In [ ]:
#@title 11. 기준실험과 변경실험의 공간지도 직접 비교
reference_experiment = "baseline" #@param {type:"string"}
reference_map_age_ka = 0.0 #@param {type:"number"}
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

ref_work=exp_root/reference_experiment
ref_output=ref_work/"output"
ref_settings=ref_work/"teaching_settings.json"
if experiment_name==reference_experiment:
    print("현재 실행이 기준실험입니다. 설정을 변경하고 실험 이름을 달리하여 실행하면 공간 비교가 가능합니다.")
elif not ref_settings.is_file() or not ref_output.is_dir():
    print(f"기준실험 '{reference_experiment}' 결과가 없습니다. 기준실험을 먼저 실행하십시오.")
else:
    ref_conf=json.loads(ref_settings.read_text(encoding="utf-8"))
    assert np.isclose(float(ref_conf["end_ka"]),float(end_ka),atol=1e-6), (
        "기준실험과 현재 실험의 계산 종료시점이 다릅니다. 같은 기간으로 실행하십시오.")
    common=[]
    for age in selected_ages:
        if all((folder/case/snapshot_dir_name(age)/"soil_depth_m.tif").exists()
               for folder in (ref_output,out_dir) for case in CASE_DIRS.values()):
            common.append(age)
    if not common:
        print("두 실험의 공통 지도 저장시점이 없습니다.")
    else:
        map_age=min(common,key=lambda a:abs(a-float(reference_map_age_ka)))
        if abs(map_age-float(reference_map_age_ka))>1e-6:
            print(f"요청 시점 대신 저장된 {map_age:g} ka 지도를 사용합니다.")
        def read_experiment_map(root,sub,ka,name):
            fn=root/sub/snapshot_dir_name(ka)/(name+".tif")
            with rasterio.open(fn) as src:
                return src.read(1,masked=True).filled(np.nan).astype(float)
        fig,axes=plt.subplots(4,3,figsize=(12,13),constrained_layout=True)
        map_stats=[]
        for i,(case,subdir) in enumerate(CASE_DIRS.items()):
            before=read_experiment_map(ref_output,subdir,map_age,"soil_depth_m")
            after=read_experiment_map(out_dir,subdir,map_age,"soil_depth_m")
            assert before.shape==after.shape
            valid=np.isfinite(before)&np.isfinite(after)
            difference=np.where(valid,after-before,np.nan)
            baseline_soil=np.ma.masked_invalid(before)
            scenario_soil=np.ma.masked_invalid(after)
            vmax=float(np.nanmax(np.r_[before[valid],after[valid]]))
            bound=max(float(np.nanmax(np.abs(difference[valid]))),1e-6)
            for j,(data,title,cmap,vmin,upper) in enumerate([
                (baseline_soil,reference_experiment,"YlOrBr",0,vmax),
                (scenario_soil,experiment_name,"YlOrBr",0,vmax),
                (np.ma.masked_invalid(difference),"변경 − 기준","coolwarm",-bound,bound),
            ]):
                ax=axes[2*i,j]
                im=ax.imshow(data,cmap=cmap,vmin=vmin,vmax=upper,interpolation="nearest")
                ax.set_title(f"{case} 토심 | {title}")
                ax.set_xticks([]);ax.set_yticks([])
                fig.colorbar(im,ax=ax,fraction=.045,pad=.03)
            sv=read_experiment_map(ref_output,subdir,map_age,"vegetation_code")
            dv=read_experiment_map(out_dir,subdir,map_age,"vegetation_code")
            valid=np.isfinite(sv)&np.isfinite(dv)
            mismatch=np.full(sv.shape,np.nan)
            mismatch[valid]=(sv[valid]!=dv[valid]).astype(float)
            veg_cmap=ListedColormap(["#2c7fb8","#2ca02c","#addd8e","#fdbf6f","#bdbdbd"])
            veg_norm=BoundaryNorm(np.arange(-.5,5.5,1),veg_cmap.N)
            for j,(arr,title) in enumerate(((sv,reference_experiment),(dv,experiment_name))):
                ax=axes[2*i+1,j]
                ax.imshow(np.ma.masked_invalid(arr),cmap=veg_cmap,norm=veg_norm,
                          interpolation="nearest")
                ax.set_title(f"{case} 식생 | {title}")
                ax.set_xticks([]);ax.set_yticks([])
            ax=axes[2*i+1,2]
            ax.imshow(np.ma.masked_invalid(mismatch),
                cmap=ListedColormap(["#1b9e77","#e31a1c"]),vmin=-.5,vmax=1.5,
                interpolation="nearest")
            ax.set_title(f"{case} 식생 | 일치(녹색)·불일치(적색)")
            ax.set_xticks([]);ax.set_yticks([])
            nn=int(valid.sum())
            map_stats.append({"model":case,"ka_bp":map_age,"valid_cells":nn,
                "same_vegetation_cells":int(np.count_nonzero(mismatch[valid]==0)),
                "vegetation_agreement_percent":float(100*np.count_nonzero(mismatch[valid]==0)/nn)})
        fig.suptitle(f"{map_age:g} ka BP | 기준실험과 변경실험의 공간 비교",fontsize=14)
        fig.legend(handles=[Patch(facecolor=veg_cmap.colors[k],label=VEG_NAMES[k])
            for k in VEG_NAMES],loc="lower center",ncol=5,bbox_to_anchor=(.5,-.01))
        output_image=report_dir/("11_scenario_maps_"+reference_experiment+"_vs_"+experiment_name+
                                 "_"+str(map_age).replace(".","p")+"ka.png")
        fig.savefig(output_image,dpi=170,bbox_inches="tight")
        plt.show()
        exp_agreement=pd.DataFrame(map_stats)
        display(exp_agreement.round(2))
        exp_agreement.to_csv(report_dir/"11_experiment_vegetation_agreement.csv",
            index=False,encoding="utf-8-sig")
        # 지도가 저장된 모든 공통 시기의 차이를 계산해 CSV로 기록
        records=[]
        for age in common:
            for case,subdir in CASE_DIRS.items():
                for var in ("soil_depth_m","elevation_m","npp","agb_reich_lai_sapwood"):
                    old=read_experiment_map(ref_output,subdir,age,var)
                    new=read_experiment_map(out_dir,subdir,age,var)
                    valid=np.isfinite(old)&np.isfinite(new)
                    delta=new[valid]-old[valid]
                    records.append({"ka_bp":age,"model":case,"variable":var,
                        "n_cells":int(valid.sum()),"baseline_mean":np.mean(old[valid]),
                        "current_mean":np.mean(new[valid]),"difference_mean":np.mean(delta),
                        "difference_p10":np.percentile(delta,10),
                        "difference_median":np.median(delta),
                        "difference_p90":np.percentile(delta,90),
                        "mean_absolute_difference":np.mean(np.abs(delta)),
                        "increased_percent":100*np.mean(delta>1e-8),
                        "decreased_percent":100*np.mean(delta< -1e-8)})
        full_exp_spatial=pd.DataFrame(records)
        full_exp_spatial.to_csv(report_dir/"12_experiment_spatial_comparison.csv",
            index=False,encoding="utf-8-sig")
        print("기준실험과 변경실험의 지도·표 저장:",report_dir)


## 12. 보고서용 파일 저장
수치자료와 그림은 매 실험마다 `student_report` 폴더와 `figures` 폴더에 저장된다. 아래 셀을 실행하면 과제에 필요한 CSV·PNG·실험설정과 **학생 보고서 작성 서식**을 ZIP으로 묶는다.

**보고서의 기본 구성:** 실험 목적과 가설 → 조정한 입력조건 → 시기별 토심·식생 지도 → 식생 면적비율과 모델 간 공간 차이 → 기준실험과 변경실험 비교 → 지형·식생 상호작용에 대한 해석 및 한계.

보고서에는 실험기간, 조정한 매개변수, 비교 대상(정적/동적 또는 서로 다른 실험), 비교 시기 및 차이의 부호를 반드시 기록한다. AGB*는 모델에서 산출한 추정지표이므로 실측 지상부 바이오매스와 구분한다.


In [ ]:
#@title 12. 과제용 CSV·지도·설정파일 압축
download_report_zip = False #@param {type:"boolean"}
from zipfile import ZipFile,ZIP_DEFLATED
# 기본 실험정보를 기록한 보고서 작성 서식
template_path=work/"student_report_template.md"
template_text=f"""# 용늪 21–0 ka 생물지형학 실습 보고서

## 1. 실험 목적 및 가설
- 비교하려는 지형·식생 과정:
- 예상한 변화와 그 근거:

## 2. 실험 조건
- 실험명: {experiment_name}
- 계산 기간: 21.0–{end_ka:g} ka BP (0.1 kyr 결합간격)
- 기후 유형: {climate_type}
- 기온 편차: {dT:+.2f} °C
- 월강수 배율: {p_factor:.3f}배
- 초기 토심 배율: {initial_soil_depth_multiplier:.3f}배
- 융기율: {uplift_m_per_kyr:g} m/kyr
- 토양생산률 계수 a: {soil_production_a:g}
- 사면수송 계수 c, d: {hillslope_eemt_c:g}, {hillslope_biomass_d:g}
- 유수침식 계수 K0: {fluvial_erodibility_k0:g}
- 비교 시기: {', '.join(f'{a:g}' for a in selected_ages)} ka BP

## 3. 시기별 공간분포
- 그림: figures/snapshot_dynamic_static_soil_depth_m.png
- 그림: figures/snapshot_dynamic_static_vegetation_code.png
- 같은 시점의 정적모델·동적모델 및 차이(동적−정적)를 비교한다.
- 차이가 크게 나타나는 격자의 위치 및 분포:

## 4. 토심·고도·식생생산량
- 표: student_report/02_selected_ages_comparison.csv
- 표: student_report/07_spatial_difference_statistics.csv
- 평균·표준편차·10/50/90백분위수와 증가/감소 면적비율:
- 평균 차이와 공간적 차이가 다르게 나타나는 이유:

## 5. 식생유형의 시기별 변화
- 그림: figures/timeseries_vegetation_ratio.png (수정된 정확한 0.1 kyr 시간축)
- 표: student_report/04_selected_ages_vegetation_ratio.csv
- 표: student_report/05_vegetation_agreement.csv
- 표: student_report/06_vegetation_cross_tabulation.csv
- 침엽수림·활엽수림·혼효림·초본·무식생의 면적비율 변화:
- 두 모델의 식생 일치율과 불일치 격자:
- 교차표의 방향은 정적모델 → 동적모델이며 시간상 식생 전환과 다르다.

## 6. 기준실험과 변경실험
- 비교 기준실험명:
- 동일 시점·동일 모델에서 비교한 토심·식생 분포:
- 비교 시 두 실험에서 달리한 변수:
- 기준실험과의 차이를 단일 원인으로 설명할 수 있는가?

## 7. 토의 및 결론
- 기후·식생·토심 및 사면과정 간 상호작용 해석:
- 대체 해석 또는 자료의 한계:
- 모형 결과는 가상 매개변수 교란에 따른 산출값이며 관측값이 아님을 명시한다.
- AGB*는 모형기반 지상부 바이오매스 추정지표로 취급한다.

## 자료 및 재현성
- 모형: PB4Studio v6.6.3-CHELSA21K (BIOME4–Pelletier)
- 모형 자료 SHA256: {checksum}
- 원본: https://github.com/physical-geogrpahy-forever/yongneup_nb
"""
template_path.write_text(template_text,encoding="utf-8")
export_zip=work/(experiment_name+"_student_report.zip")
with ZipFile(export_zip,"w",ZIP_DEFLATED) as archive:
    for folder in (report_dir,figure_dir):
        for file in sorted(folder.glob("*")):
            if file.is_file() and file.suffix.lower() in (".csv",".png"):
                archive.write(file,file.relative_to(out_dir))
    for file in (work/"teaching_settings.json",work/"run_config.json",template_path):
        if file.is_file():
            archive.write(file,file.name)
print("보고서 자료 ZIP:",export_zip)
if download_report_zip:
    try:
        from google.colab import files
        files.download(str(export_zip))
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(export_zip)))
else:
    print("내려받으려면 위의 download_report_zip에 체크하고 이 셀을 다시 실행하십시오.")
